# 🧠 Re:Learn — AI/ML Hackathon Training Pipeline
### Fine-Tuning FLAN-T5 for Code Misconception Diagnosis & Socratic Correction

This notebook fine-tunes **Google FLAN-T5** (`google/flan-t5-small`), an instruction-tuned sequence-to-sequence transformer, to inspect student code submissions, detect cognitive misconceptions, formulate pedagogical interventions, and output corrected code.

**Hardware**: Google Colab Free Tier with T4 GPU (`Runtime` -> `Change runtime type` -> `T4 GPU`).

## Step 1: Install Required AI/ML Libraries

In [ ]:
!pip install -q transformers datasets accelerate torch evaluate scikit-learn matplotlib fastapi uvicorn nest_asyncio pyngrok sentencepiece

import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU not enabled. Please go to Runtime -> Change runtime type -> Select T4 GPU.")

## Step 2: Build the Misconception & Correction Dataset

In [ ]:
import json
import random

DATASET = [
    {
        "problem": "Write a function is_even(n) returning True if n is even, False otherwise.",
        "student_code": "def is_even(n):\n    if n % 2 = 0:\n        return True\n    return False",
        "is_correct": False,
        "misconception": "Assignment operator '=' used in comparison context",
        "explanation": "A single '=' assigns a value to a variable, whereas '==' compares values. Inside an if condition, Python expects a boolean expression.",
        "intervention": "Notice the difference between creating a variable and checking its value. How do you test equality in Python?",
        "fixed_code": "def is_even(n):\n    if n % 2 == 0:\n        return True\n    return False"
    },
    {
        "problem": "Write a function is_even(n) returning True if n is even, False otherwise.",
        "student_code": "def is_even(n):\n    return n % 2 == 0",
        "is_correct": True,
        "misconception": "None",
        "explanation": "The solution directly and accurately evaluates the boolean condition of remainder zero.",
        "intervention": "Great work! The comparison evaluates directly to True or False.",
        "fixed_code": "def is_even(n):\n    return n % 2 == 0"
    },
    {
        "problem": "Check if count reached 10 in a loop.",
        "student_code": "while count = 10:\n    print(count)\n    count += 1",
        "is_correct": False,
        "misconception": "Assignment operator '=' used in comparison context",
        "explanation": "Inside loop conditions, assignment statements are invalid syntax. Comparison operators must be used.",
        "intervention": "Use comparative equality '==' or relational operators to test loop conditions.",
        "fixed_code": "while count < 10:\n    print(count)\n    count += 1"
    },
    {
        "problem": "Write a function is_even(n) returning True if n is even, False otherwise using bitwise operators.",
        "student_code": "def is_even(n):\n    return (n & 1) == 0",
        "is_correct": True,
        "misconception": "None",
        "explanation": "Even integers have 0 as their lowest bit; bitwise AND with 1 verifies even parity optimally.",
        "intervention": "Excellent optimization using bitwise operations.",
        "fixed_code": "def is_even(n):\n    return (n & 1) == 0"
    },
    {
        "problem": "Write a function is_even(n) returning True if n is even, False otherwise.",
        "student_code": "def is_even(n):\n    if n % 2:\n        return True\n    return False",
        "is_correct": False,
        "misconception": "Truthy remainder inverted logic",
        "explanation": "In Python, non-zero integers evaluate to True. When n is odd, n % 2 is 1 (True). This reverses the logic.",
        "intervention": "What is 4 % 2 vs 5 % 2? In Python, does 0 evaluate to True or False in an if condition?",
        "fixed_code": "def is_even(n):\n    if not (n % 2):\n        return True\n    return False"
    },
    {
        "problem": "Write a function append_item(val, container=[]) that adds val to container.",
        "student_code": "def append_item(val, container=[]):\n    container.append(val)\n    return container",
        "is_correct": False,
        "misconception": "Mutable default argument state retention",
        "explanation": "Default arguments are evaluated once at function definition time. A mutable list is shared across calls.",
        "intervention": "What happens if you invoke append_item without passing a container twice? Use None as default.",
        "fixed_code": "def append_item(val, container=None):\n    if container is None:\n        container = []\n    container.append(val)\n    return container"
    },
    {
        "problem": "Find the maximum element in a list of numbers.",
        "student_code": "def find_max(numbers):\n    max_val = numbers[0]\n    for i in range(1, len(numbers) + 1):\n        if numbers[i] > max_val:\n            max_val = numbers[i]\n    return max_val",
        "is_correct": False,
        "misconception": "Index out of bounds / range upper bound inclusive assumption",
        "explanation": "Python lists are 0-indexed up to len(numbers) - 1. range(1, len(numbers) + 1) accesses index len(numbers), raising IndexError.",
        "intervention": "What is the highest valid index in a 3-element list? Does range(1, len(numbers)) suffice?",
        "fixed_code": "def find_max(numbers):\n    max_val = numbers[0]\n    for i in range(1, len(numbers)):\n        if numbers[i] > max_val:\n            max_val = numbers[i]\n    return max_val"
    },
    {
        "problem": "Write a function isValidToken(role, tier) verifying role is 'admin' and tier is integer 1.",
        "student_code": "function isValidToken(role, tier) {\n    if (role == 'admin' && tier == 1) {\n        return true;\n    }\n    return false;\n}",
        "is_correct": False,
        "misconception": "Loose equality (==) allows implicit type coercion",
        "explanation": "Loose equality '==' coerces string '1' to number 1, which permits unauthorized string payloads.",
        "intervention": "Which JavaScript operator checks both the value and the exact type without coercion?",
        "fixed_code": "function isValidToken(role, tier) {\n    return role === 'admin' && tier === 1;\n}"
    },
    {
        "problem": "Increment a counter tracked outside the function.",
        "student_code": "total = 0\ndef add_to_total(amount):\n    total += amount\n    return total",
        "is_correct": False,
        "misconception": "Local assignment shadowing without global declaration",
        "explanation": "Assigning to 'total' inside the function makes it local, but reading it before assignment raises UnboundLocalError.",
        "intervention": "Use the 'global' keyword to explicitly bind total to the outer scope.",
        "fixed_code": "total = 0\ndef add_to_total(amount):\n    global total\n    total += amount\n    return total"
    },
    {
        "problem": "Add two numbers entered as user input strings.",
        "student_code": "def add_inputs(a, b):\n    return a + b",
        "is_correct": False,
        "misconception": "String concatenation vs numeric addition",
        "explanation": "Applying '+' between strings concatenates them rather than computing arithmetic addition.",
        "intervention": "Convert input strings to integers using int().",
        "fixed_code": "def add_inputs(a, b):\n    return int(a) + int(b)"
    },
    {
        "problem": "Duplicate a 2D matrix without modifying the original.",
        "student_code": "def duplicate_grid(grid):\n    new_grid = list(grid)\n    new_grid[0][0] = 999\n    return new_grid",
        "is_correct": False,
        "misconception": "Shallow copy mutates nested sublists",
        "explanation": "list(grid) creates a shallow copy where inner lists are still shared references.",
        "intervention": "Use copy.deepcopy() to recursively clone nested lists.",
        "fixed_code": "import copy\ndef duplicate_grid(grid):\n    new_grid = copy.deepcopy(grid)\n    new_grid[0][0] = 999\n    return new_grid"
    },
    {
        "problem": "Calculate factorial of n recursively.",
        "student_code": "def factorial(n):\n    if n <= 1:\n        return 1\n    factorial(n - 1) * n",
        "is_correct": False,
        "misconception": "Omitted return keyword in recursive step",
        "explanation": "Without the return keyword on the recursive expression, the function returns None.",
        "intervention": "Ensure every recursive branch returns its result.",
        "fixed_code": "def factorial(n):\n    if n <= 1:\n        return 1\n    return factorial(n - 1) * n"
    }
]

print(f"✅ Loaded {len(DATASET)} core misconception samples.")

## Step 3: Format and Tokenize for Google FLAN-T5

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

formatted_samples = []
for item in DATASET:
    inp = f"Diagnose misconception | Problem: {item['problem']} | Code: {item['student_code']}"
    tgt = json.dumps({
        "is_correct": item["is_correct"],
        "misconception": item["misconception"],
        "explanation": item["explanation"],
        "intervention": item["intervention"],
        "fixed_code": item["fixed_code"]
    })
    formatted_samples.append({"input_text": inp, "target_text": tgt})

# Augment data to allow robust convergence
data_pool = formatted_samples * 12
random.seed(42)
random.shuffle(data_pool)

split = int(0.85 * len(data_pool))
train_ds = Dataset.from_list(data_pool[:split])
val_ds = Dataset.from_list(data_pool[split:])

def tokenize_fn(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(text_target=examples["target_text"], max_length=256, truncation=True, padding="max_length")
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_ds.map(tokenize_fn, batched=True)
tokenized_val = val_ds.map(tokenize_fn, batched=True)
print(f"✅ Tokenization complete! Train: {len(tokenized_train)} | Val: {len(tokenized_val)}")

## Step 4: Fine-Tune FLAN-T5 Model with Seq2SeqTrainer

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

training_args = Seq2SeqTrainingArguments(
    output_dir="./flant5_misconception_model",
    eval_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_total_limit=1,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
)

print("🚀 Starting Training on GPU...")
trainer.train()
model.save_pretrained("./relearn_trained_model")
tokenizer.save_pretrained("./relearn_trained_model")
print("✅ Model trained and weights saved to ./relearn_trained_model")

## Step 5: Plot Training Loss & Accuracy Metrics

In [ ]:
import matplotlib.pyplot as plt

losses = [x["loss"] for x in trainer.state.log_history if "loss" in x]

plt.figure(figsize=(8, 4))
plt.plot(losses, marker="o", color="#2563eb", linewidth=2)
plt.title("Re:Learn FLAN-T5 Training Loss Convergence")
plt.xlabel("Training Steps")
plt.ylabel("Cross Entropy Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## Step 6: Test Inference on Unseen Student Submissions

In [ ]:
def diagnose(problem: str, code: str):
    prompt = f"Diagnose misconception | Problem: {problem} | Code: {code}"
    inputs = tokenizer(prompt, return_tensors="pt", max_length=256, truncation=True).to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_length=256,
            num_beams=3,
            early_stopping=True
        )
    
    output_str = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    try:
        return json.loads(output_str)
    except Exception:
        return {"raw_output": output_str}

# Test 1: Buggy code
test_code = "def is_even(n):\n    if n % 2 = 0:\n        return True\n    return False"
print("Input Code:")
print(test_code)
print("\nAI ML Model Diagnosis:")
print(json.dumps(diagnose("Check if even", test_code), indent=2))

## Step 7: Launch Live REST API (Localtunnel / Ngrok)
Run this cell to start an active API server that connects directly to the React web UI.

In [ ]:
import nest_asyncio
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import threading

app = FastAPI(title="Re:Learn Model API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class DiagnoseRequest(BaseModel):
    problem: str = ""
    code: str

@app.get("/")
def health():
    return {"status": "online", "model": "FLAN-T5-Misconception-Diagnoser"}

@app.post("/diagnose")
def api_diagnose(req: DiagnoseRequest):
    return diagnose(req.problem, req.code)

nest_asyncio.apply()
server_thread = threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000), daemon=True)
server_thread.start()

print("Server running on port 8000!")
print("Starting localtunnel...")
!npx localtunnel --port 8000